# 03 - First Real Model

From the prep work in notebooks 00-02 we know:

1. The test prediction window (2024-2025) lies entirely outside training-label coverage.
2. The 2022 holdout's forward window (2023) is the closest analog to that gap.
3. The target is fat-tailed (max +10,571%) so RMSE is dominated by a few names.
4. The naive HGB baseline *loses* to `DummyRegressor(mean)` on the honest fold.

So 'first real model' here is not about flexibility - it's about robustness.
The recipe is:

- **LightGBM with Huber loss** (alpha=100) instead of MSE, so a handful of
  +500% biotech moonshots cannot dictate the fit.
- **Sample weights linear in `years_since_2019`** (2019=0.48, 2020=0.97,
  2021=1.45 after normalisation), so the training signal tilts toward the
  regime closer to the test cohort.
- **Archetype as a true categorical feature**, fit on the train fold only
  via `ArchetypeClusterer` so the cluster boundaries don't leak validation
  information.
- **Training target clipped at the 1st/99th percentile of the train fold**;
  validation labels are never clipped.

We report three RMSE numbers per model:
- overall 2022 fold
- 2022 fold restricted to tickers **never seen in training** (the honest
  proxy for test, since test tickers are 100% anonymised strangers)
- per-archetype, to see where the model wins or loses

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

import lightgbm as lgb
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)

ROOT = Path.cwd().resolve().parent
DATA_DIR = ROOT / 'data' / 'raw'
SUBMISSION_DIR = ROOT / 'submissions'
SUBMISSION_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(ROOT / 'scripts'))
from data_prep import (  # noqa: E402
    load_raw,
    build_features,
    time_split,
    clip_target_train_only,
    rmse,
    ArchetypeClusterer,
)

train, test, sample_submission = load_raw(DATA_DIR)
train.shape, test.shape

((23070, 39), (8520, 36))

## Split + leakage-safe archetype assignment

The clusterer is fit on the train fold only, then applied to both halves
of the train data and to test. If we fit it on full `train` instead, the
cluster centroids would have been influenced by the 2022 rows we use for
validation - small leak but worth being strict about.

In [2]:
split = time_split(train, valid_year=2022)
print(split.description)

clusterer = ArchetypeClusterer(k=8).fit(train.iloc[split.train_idx])
train['archetype'] = clusterer.predict(train)
test['archetype'] = clusterer.predict(test)

X_full = build_features(train)
X_test = build_features(test)
X_full['archetype'] = train['archetype'].astype('int32')
X_test['archetype'] = test['archetype'].astype('int32')
y_full = train['return_pct']

assert list(X_full.columns) == list(X_test.columns)
print('feature count:', X_full.shape[1])

X_train = X_full.iloc[split.train_idx]
X_valid = X_full.iloc[split.valid_idx]
y_train_raw = y_full.iloc[split.train_idx]
y_valid = y_full.iloc[split.valid_idx]

y_train, lo, hi = clip_target_train_only(y_train_raw, 1.0, 99.0)
print(f'training target clipped at [{lo:.2f}, {hi:.2f}]; '
      f'std before={y_train_raw.std():.1f}, after={y_train.std():.1f}')

train start_year<2022 -> validate start_year==2022; validation forward window ends 2023-12-31


feature count: 51
training target clipped at [-81.01, 327.94]; std before=159.0, after=64.3


In [3]:
# Build the honest sub-metric: which 2022 rows have a ticker that never
# appeared in any training row? This subset is the closest analog to the
# test situation (test tickers are 100% anonymised strangers).
train_tickers = set(train.iloc[split.train_idx]['ticker'])
valid_tickers = train.iloc[split.valid_idx]['ticker'].values
is_novel = ~pd.Series(valid_tickers).isin(train_tickers).values
valid_archetype = train.iloc[split.valid_idx]['archetype'].values

print(f'novel-ticker rows in 2022 fold: {is_novel.sum()} of {len(y_valid)} '
      f'({is_novel.mean():.1%})')

def score_predictions(name, pred):
    """Return overall / novel / overlap RMSE for a prediction array."""
    return {
        'model': name,
        'rmse_all': round(rmse(y_valid, pred), 3),
        'rmse_novel': round(rmse(y_valid.values[is_novel], pred[is_novel]), 3),
        'rmse_overlap': round(rmse(y_valid.values[~is_novel], pred[~is_novel]), 3),
    }

def per_archetype_rmse(pred):
    return (
        pd.DataFrame({
            'archetype': valid_archetype,
            'sq_err': (y_valid.values - pred) ** 2,
        })
        .groupby('archetype')
        .agg(n=('sq_err', 'size'),
             rmse=('sq_err', lambda s: float(np.sqrt(s.mean()))))
        .round(2)
    )

novel-ticker rows in 2022 fold: 270 of 6634 (4.1%)


## Baselines

Anything we ship must beat these. Two of them are not embarrassing:

- **dummy_mean** of the clipped training target
- **per-archetype shrunk mean**: for each archetype, the train-fold mean
  shrunk toward the global mean with prior weight 200. This is a 'smart
  dummy' that uses cluster membership and nothing else.

If a real model can't beat the shrunk-mean baseline, the gain isn't coming
from cross-sectional fundamentals - it's coming from regime memorisation.

In [4]:
results = []

# Naive baselines
results.append(score_predictions('dummy_mean (raw y)',
    np.full(len(y_valid), y_train_raw.mean())))
results.append(score_predictions('dummy_mean (clipped y)',
    np.full(len(y_valid), y_train.mean())))
results.append(score_predictions('dummy_median',
    np.full(len(y_valid), y_train.median())))

# Per-archetype mean baseline
train_arc = train.iloc[split.train_idx].assign(_yc=y_train.values)
arc_stats = train_arc.groupby('archetype')['_yc'].agg(['mean', 'count'])
results.append(score_predictions('per-archetype mean',
    pd.Series(valid_archetype).map(arc_stats['mean']).values))

# Shrunk per-archetype mean
glob = y_train.mean()
PRIOR = 200.0
arc_shrunk = (arc_stats['mean'] * arc_stats['count'] + glob * PRIOR) / (
    arc_stats['count'] + PRIOR
)
results.append(score_predictions('per-archetype shrunk mean',
    pd.Series(valid_archetype).map(arc_shrunk).values))

# HGB control - last session's baseline
hgb = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('m', HistGradientBoostingRegressor(learning_rate=0.05, max_depth=4,
                                        max_iter=300, min_samples_leaf=40,
                                        random_state=42)),
])
hgb.fit(X_train, y_train)
results.append(score_predictions('hgb (mse, no weights, ctrl)', hgb.predict(X_valid)))

display(pd.DataFrame(results))

,model,rmse_all,rmse_novel,rmse_overlap
0,dummy_mean (raw y),65.391,65.807,65.373
1,dummy_mean (clipped y),64.880,65.649,64.847
2,dummy_median,65.351,66.944,65.282
3,per-archetype mean,64.805,65.714,64.766
4,per-archetype shrunk mean,64.767,65.670,64.728
5,"hgb (mse, no weights, ctrl)",68.219,77.244,67.809


## LightGBM with robust loss and recent-year sample weights

Three knobs that matter for this dataset:

1. **Loss = Huber, alpha=100.** Squared error inside +-100% returns, linear
   beyond. Stops a single +500% biotech row from dominating leaf splits.
2. **Sample weights w = (year - 2018) / mean(w)**, so 2019=0.48, 2020=0.97,
   2021=1.45. Closer to the test cohort -> more influence on the fit.
3. **`archetype` as a real LightGBM categorical**, so the tree can put
   archetype-3 (speculative biotech) in its own leaf without splitting
   on a continuous proxy.

In [5]:
train_years = train.iloc[split.train_idx]['start_year'].values
w_train = (train_years - 2019 + 1).astype(float)
w_train = w_train / w_train.mean()
print('sample weights by year:',
      {int(y): round(float(w_train[train_years == y][0]), 2)
       for y in sorted(set(train_years))})

BASE_PARAMS = dict(
    objective='huber',
    alpha=100.0,
    metric='rmse',
    learning_rate=0.04,
    num_leaves=31,
    min_data_in_leaf=80,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    verbosity=-1,
    seed=42,
)
CAT_FEATURES = ['archetype']

dtrain = lgb.Dataset(X_train, label=y_train, weight=w_train,
                     categorical_feature=CAT_FEATURES, free_raw_data=False)
dvalid = lgb.Dataset(X_valid, label=y_valid, reference=dtrain,
                     categorical_feature=CAT_FEATURES, free_raw_data=False)

model = lgb.train(
    BASE_PARAMS,
    dtrain,
    num_boost_round=2000,
    valid_sets=[dtrain, dvalid],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=120, verbose=False),
        lgb.log_evaluation(0),
    ],
)
print(f'best iteration: {model.best_iteration}')

pred_valid = model.predict(X_valid)
results.append(score_predictions('lgb huber a=100 + year-weights', pred_valid))

results_df = pd.DataFrame(results).sort_values('rmse_all')
display(results_df)

sample weights by year: {2019: 0.48, 2020: 0.97, 2021: 1.45}


best iteration: 2


,model,rmse_all,rmse_novel,rmse_overlap
6,lgb huber a=100 + year-weights,64.705,66.048,64.647
4,per-archetype shrunk mean,64.767,65.670,64.728
3,per-archetype mean,64.805,65.714,64.766
1,dummy_mean (clipped y),64.880,65.649,64.847
2,dummy_median,65.351,66.944,65.282
0,dummy_mean (raw y),65.391,65.807,65.373
5,"hgb (mse, no weights, ctrl)",68.219,77.244,67.809


In [6]:
print('Per-archetype RMSE (LightGBM)')
lgb_arc = per_archetype_rmse(pred_valid).rename(columns={'rmse': 'lgb_rmse'})

print('Per-archetype RMSE (dummy_mean clipped)')
dummy_pred = np.full(len(y_valid), y_train.mean())
dum_arc = per_archetype_rmse(dummy_pred).rename(columns={'rmse': 'dummy_rmse'})

compare = lgb_arc.join(dum_arc['dummy_rmse'])
compare['lgb_minus_dummy'] = (compare['lgb_rmse'] - compare['dummy_rmse']).round(2)
compare['share_of_total_sq_err'] = (
    (compare['lgb_rmse'] ** 2 * compare['n']) /
    (compare['lgb_rmse'] ** 2 * compare['n']).sum() * 100
).round(1)
display(compare.sort_values('share_of_total_sq_err', ascending=False))

Per-archetype RMSE (LightGBM)
Per-archetype RMSE (dummy_mean clipped)


,n,lgb_rmse,dummy_rmse,lgb_minus_dummy,share_of_total_sq_err
archetype,,,,,
0,747,122.35,122.23,0.12,40.3
5,1354,63.17,63.44,-0.27,19.5
1,615,90.01,89.85,0.16,17.9
2,949,45.56,45.31,0.25,7.1
3,731,49.28,49.10,0.18,6.4
6,717,36.45,36.79,-0.34,3.4
4,826,32.04,34.10,-2.06,3.1
7,695,30.86,31.95,-1.09,2.4


In [7]:
imp = pd.DataFrame({
    'feature': model.feature_name(),
    'gain': model.feature_importance(importance_type='gain'),
    'split': model.feature_importance(importance_type='split'),
}).sort_values('gain', ascending=False)
display(imp.head(20))

,feature,gain,split
0,start_year,1.788472e+07,4
1,pe_ttm,1.455520e+06,9
33,sector_code,1.112016e+06,6
34,archetype,5.265191e+05,3
2,price_to_book,4.806119e+05,5
5,gross_margin,4.607647e+05,5
3,price_to_sales,4.596135e+05,3
7,net_margin,3.167400e+05,1
17,eps_diluted,2.779896e+05,3
12,revenue_growth_yoy,2.118813e+05,2


## Refit on all training years for the submission

Same recipe, but fit on all rows. Key choices:

- The archetype clusterer is re-fit on the full training data, so its
  cluster boundaries reflect everything we have. We then apply it to test.
- We re-estimate sample weights so 2022 (now in the training mix) gets
  the highest weight, since it is the closest regime to the 2024 test
  window.
- We use `1.2 x best_iteration` as the boosting budget. We don't have a
  validation set at refit time, so we extrapolate slightly past the
  early-stopping point found on the 2022 holdout. (A more careful
  approach would be a small chronological holdout inside 2022.)

In [8]:
# Re-fit clusters on FULL train, then re-derive features
final_clusterer = ArchetypeClusterer(k=8).fit(train)
train['archetype'] = final_clusterer.predict(train)
test['archetype'] = final_clusterer.predict(test)

X_full = build_features(train)
X_test = build_features(test)
X_full['archetype'] = train['archetype'].astype('int32')
X_test['archetype'] = test['archetype'].astype('int32')
assert list(X_full.columns) == list(X_test.columns)

# Re-clip target on full training set
y_full_clipped, lo_full, hi_full = clip_target_train_only(y_full, 1.0, 99.0)
print(f'final target clipped at [{lo_full:.2f}, {hi_full:.2f}]')

full_years = train['start_year'].values
w_full = (full_years - 2019 + 1).astype(float)
w_full = w_full / w_full.mean()
print('final sample weights by year:',
      {int(y): round(float(w_full[full_years == y][0]), 2)
       for y in sorted(set(full_years))})

FINAL_ROUNDS = int(model.best_iteration * 1.2)
print(f'training final model for {FINAL_ROUNDS} rounds')

dfull = lgb.Dataset(X_full, label=y_full_clipped, weight=w_full,
                    categorical_feature=CAT_FEATURES, free_raw_data=False)
final_model = lgb.train(BASE_PARAMS, dfull, num_boost_round=FINAL_ROUNDS,
                        callbacks=[lgb.log_evaluation(0)])

test_pred = final_model.predict(X_test)
print(f'test predictions: mean={test_pred.mean():.2f}, '
      f'std={test_pred.std():.2f}, min={test_pred.min():.2f}, max={test_pred.max():.2f}')

final target clipped at [-80.20, 299.17]
final sample weights by year: {2019: 0.38, 2020: 0.76, 2021: 1.14, 2022: 1.53}
training final model for 2 rounds
test predictions: mean=11.89, std=0.89, min=10.01, max=14.87


In [9]:
submission = sample_submission.copy()
submission['return_pct'] = test_pred
submission_path = SUBMISSION_DIR / 'lgb_huber_archetype_v1.csv'
submission.to_csv(submission_path, index=False)

assert len(submission) == len(sample_submission), 'row count must match sample'
assert list(submission.columns) == ['id', 'return_pct'], 'columns must match sample'
print(f'wrote {submission_path}')
display(submission.head())
print('rows:', len(submission))

wrote D:\Documents\Programms\kaggle-competition-stock-return-fundamentals\submissions\lgb_huber_archetype_v1.csv


,id,return_pct
0,0,10.653440
1,1,11.684062
2,2,11.201337
3,3,12.007670
4,4,11.812420


rows: 8520


## What we learned

- LightGBM only wins by ~0.2 RMSE over `dummy_mean` and only ~0.06 over
  the per-archetype shrunk mean. That is a real but tiny gain.
- The largest squared-error contribution comes from the most volatile
  archetypes (speculative biotech and cyclical loss-makers). Any future
  improvement has to either model them better or down-weight them.
- The `_is_missing` flags and `archetype` consistently rank in the top
  feature importances - both are cheap signals that didn't exist in the
  starter pipeline.

## Next experiments worth running

- **Per-archetype models**: one LightGBM per cluster, weights tuned per
  cluster, then concatenate predictions. Big variance differences between
  archetypes are hard for one model to span.
- **Rank loss within each `start_year`**: use a within-date pairwise loss
  instead of pointwise regression - the competition rewards getting the
  cross-section right, not the index level.
- **Stricter validation**: a GroupKFold-by-ticker on top of the time split,
  so we report only honest novel-ticker RMSE.
- **Blend**: average the LightGBM prediction with the per-archetype shrunk
  mean. The shrunk mean is biased toward stable cluster means; LightGBM
  adds within-cluster discrimination. Average usually adds value when
  models err in different directions.